# 01 - Classification: Opportunity Result

Benchmark notebook for the `Opportunity Result` target.

Model families compared include statsmodels classifiers, regularized logistic models, binned variants, and XGBoost.

Data source: `../../data/intermediate/df_train_stratified.parquet`.


In [1]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from binning import NamedBinningProcess, DataFrameScaler
from statsmodels_api import StatsModelsClassifier

import statsmodels.api as sm
from tqdm.auto import tqdm




pd.set_option('display.max_columns', 200)

/Users/goviedb/Development/mib_cars_analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet('../../data/intermediate/df_train_stratified.parquet')
print('shape:', df.shape)

shape: (53073, 37)


In [3]:
# Editable feature list (all enabled by default)
FEATURES = [
    'Supplies Group',
    'Supplies Subgroup',
    'Region',
    'Route To Market',
    'Elapsed Days In Sales Stage',
    'Sales Stage Change Count',
    'Total Days Identified Through Closing',
    'Total Days Identified Through Qualified',
    #'Opportunity Amount USD',
    'Client Size By Revenue (USD)',
    'Client Size By Employee Count',
    'Revenue From Client Past Two Years (USD)',
    'Competitor Type',
    'Ratio Days Identified To Total Days',
    'Ratio Days Validated To Total Days',
    'Ratio Days Qualified To Total Days',
    #'Deal Size Category (USD)',
    #'total_days_zero',
    #'opportunity_amount_weirdness',
    #'flag_0_days',
    #'flag_ratio_problem',
    #'flag_zero_opportunity_amount',
    #'flag_outlier_opportunity_amount',
    #'flag_outlier_total_days',
    #'flag_totally_repeated_row',
    #'flag_partially_repeated_row',
    #'partial_repeat_is_latest_id_appearance',
    #'flag_only_identified',
    #'flag_weirdness_over_75th_pct',
    #'problem_count',
]

TARGET = 'Opportunity Result Bool'

missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()
print('n_features:', len(FEATURES))
print('target mean (Won rate):', y.mean())

n_features: 15
target mean (Won rate): 0.22725302884705972


In [4]:
#X["flag_0_days"] = (X["Total Days Identified Through Closing"] == 0).astype(str)
#X["flag_ratio_problem"] = (X["Total Days Identified Through Closing"] == 0).astype(str)


In [5]:
X_cols = list(X.columns)
X_categorical_cols = [
    'Supplies Group',
    'Supplies Subgroup',
    'Region',
    'Route To Market',
    'Competitor Type',
    'Client Size By Revenue (USD)',
    'Client Size By Employee Count',
    'Revenue From Client Past Two Years (USD)',
#    'flag_0_days',
#    'flag_ratio_problem',
]
X_numerical_cols = [c for c in X_cols if c not in X_categorical_cols]

In [6]:
def get_classic_preprocessor(scaler=StandardScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols):
    return ColumnTransformer([
        ('numerical', Pipeline([
            ("imputer", SimpleImputer(strategy=imputer_strategy_numerical)),
            ("scaler", scaler)
        ]), numerical_cols),
        ('categorical', Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
            ("ohe", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ]).set_output(transform="pandas")

In [7]:
experiment_grid = {
    "classic_logit_standard_scaler": Pipeline([
        ("preprocessing",get_classic_preprocessor(scaler=StandardScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', StatsModelsClassifier(sm.Logit))
    ]),
    "classic_probit_standard_scaler": Pipeline([
        ("preprocessing", get_classic_preprocessor(scaler=StandardScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', StatsModelsClassifier(sm.Probit))
    ]),
    "classic_logit_robust_scaler": Pipeline([
        ("preprocessing", get_classic_preprocessor(scaler=RobustScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', StatsModelsClassifier(sm.Logit))
    ]),
    "classic_probit_robust_scaler": Pipeline([
        ("preprocessing", get_classic_preprocessor(scaler=RobustScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', StatsModelsClassifier(sm.Probit))
    ]),
    "logit_elasticnet_robust_scaler": Pipeline([
        ("preprocessing", get_classic_preprocessor(scaler=RobustScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', LogisticRegression(solver='saga', l1_ratio=0.5, max_iter=10000))
    ]),
    "logit_elasticnet_standard_scaler": Pipeline([
        ("preprocessing", get_classic_preprocessor(scaler=StandardScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', LogisticRegression(solver='saga', l1_ratio=0.5, max_iter=10000))
    ]),
    "probit_binned_ohe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="bin_ohe"
        )),
        ('model', StatsModelsClassifier(sm.Probit))
    ]),
    "logit_binned_ohe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="bin_ohe"
        )),
        ('model', StatsModelsClassifier(sm.Logit))
    ]),
    "logit_elasticnet_binned_ohe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="bin_ohe"
        )),
        ('model', LogisticRegression(solver='saga', l1_ratio=0.5, max_iter=10000))
    ]),
    "probit_binned_woe_robust_scaler": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="woe"
        )),
        ('scaler', DataFrameScaler(RobustScaler())),
        ('model', StatsModelsClassifier(sm.Probit))
    ]),
    "logit_binned_woe_robust_scaler": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="woe"
        )),
        ('scaler', DataFrameScaler(RobustScaler())),
        ('model', StatsModelsClassifier(sm.Logit))
    ]),
    "probit_binned_woe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="woe"
        )),
        #('scaler', DataFrameScaler(RobustScaler())),
        ('model', StatsModelsClassifier(sm.Probit))
    ]),
    "logit_binned_woe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="woe"
        )),
        #('scaler', DataFrameScaler(RobustScaler())),
        ('model', StatsModelsClassifier(sm.Logit))
    ]),
    "logit_elastic_binned_woe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="woe"
        )),
        #('scaler', DataFrameScaler(RobustScaler())),
        ('model', LogisticRegression(solver='saga', l1_ratio=0.5, max_iter=10000))
    ]),
    "xgboost": Pipeline([
        ('preprocessing', get_classic_preprocessor(scaler=None, imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', XGBClassifier(use_label_encoder=False, eval_metric='logloss'))
    ])
}

In [8]:
cv = 5
skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

results = []

In [9]:
outer_bar = tqdm(experiment_grid.items(), total=len(experiment_grid), desc="Experiments", position=0)

for exp_name, pipeline in outer_bar:
    outer_bar.set_postfix_str(exp_name)

    inner_bar = tqdm(
        enumerate(skf.split(X, y), start=1),
        total=cv,
        desc=f"{exp_name} folds",
        leave=False,
        position=1
    )

    for fold, (train_idx, valid_idx) in inner_bar:
        X_train = X.iloc[train_idx] if hasattr(X, "iloc") else X[train_idx]
        X_valid = X.iloc[valid_idx] if hasattr(X, "iloc") else X[valid_idx]

        y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]
        y_valid = y.iloc[valid_idx] if hasattr(y, "iloc") else y[valid_idx]

        model = clone(pipeline)

        try:
            model.fit(X_train, y_train)

            # adjust if your wrapper does not support predict_proba
            y_proba = model.predict_proba(X_valid)[:, 1]
            y_pred = (y_proba >= 0.5).astype(int)

            results.append({
                "experiment": exp_name,
                "fold": fold,
                "roc_auc": roc_auc_score(y_valid, y_proba),
                "pr_auc": average_precision_score(y_valid, y_proba),
                "accuracy": accuracy_score(y_valid, y_pred),
                "f1": f1_score(y_valid, y_pred, zero_division=0),
                "status": "ok",
                "error": None,
            })

        except Exception as e:
            results.append({
                "experiment": exp_name,
                "fold": fold,
                "roc_auc": np.nan,
                "pr_auc": np.nan,
                "accuracy": np.nan,
                "f1": np.nan,
                "status": "failed",
                "error": str(e),
            })

Experiments:   0%|          | 0/15 [00:00<?, ?it/s]

Experiments:   0%|          | 0/15 [00:00<?, ?it/s, classic_logit_standard_scaler]

classic_logit_standard_scaler folds:   0%|          | 0/5 [00:00<?, ?it/s]

classic_logit_standard_scaler folds:  20%|██        | 1/5 [00:01<00:04,  1.16s/it]

Optimization terminated successfully.
         Current function value: 0.364942
         Iterations 8


...

xgboost folds:  80%|████████  | 4/5 [00:08<00:02,  2.09s/it]

/Users/goviedb/Development/mib_cars_analysis/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [00:31:09] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


xgboost folds: 100%|██████████| 5/5 [00:09<00:00,  1.99s/it]

Experiments: 100%|██████████| 15/15 [03:44<00:00, 10.90s/it, xgboost]

Experiments: 100%|██████████| 15/15 [03:44<00:00, 14.98s/it, xgboost]

In [10]:
results_df = pd.DataFrame(results)

summary_df = (
    results_df
    .groupby("experiment", dropna=False)
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        n_failed=("status", lambda s: (s == "failed").sum()),
    )
    .sort_values("roc_auc_mean", ascending=False)
    .reset_index()
)

results_df, summary_df

(                       experiment  fold   roc_auc    pr_auc  accuracy  \
 0   classic_logit_standard_scaler     1  0.871232  0.680131  0.829016   
 1   classic_logit_standard_scaler     2  0.858954  0.664845  0.826566   
 2   classic_logit_standard_scaler     3  0.871635  0.685353  0.831371   
 3   classic_logit_standard_scaler     4  0.864924  0.660839  0.826644   
 4   classic_logit_standard_scaler     5  0.870835  0.681718  0.828811   
 ..                            ...   ...       ...       ...       ...   
 70                        xgboost     1  0.924998  0.814766  0.883749   
 71                        xgboost     2  0.921457  0.808627  0.876213   
 72                        xgboost     3  0.930026  0.820784  0.886858   
 73                        xgboost     4  0.923584  0.810998  0.878933   
 74                        xgboost     5  0.927840  0.816654  0.881854   
 
           f1 status error  
 0   0.547269     ok  None  
 1   0.541012     ok  None  
 2   0.555611     ok  N

In [11]:
summary_df

,experiment,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,accuracy_mean,accuracy_std,f1_mean,f1_std,n_failed
0,xgboost,0.925581,0.003398,0.814365,0.004766,0.881522,0.004136,0.718448,0.010749,0
1,probit_binned_ohe,0.881099,0.004329,0.699451,0.005430,0.840842,0.002957,0.588763,0.009848,0
2,logit_binned_ohe,0.881053,0.004400,0.700006,0.005493,0.841935,0.003374,0.595983,0.010938,0
3,logit_elasticnet_binned_ohe,0.881040,0.004398,0.700108,0.005483,0.841859,0.003101,0.594316,0.010014,0
4,probit_binned_woe,0.877249,0.003758,0.692510,0.003675,0.837130,0.002506,0.563799,0.009740,0
5,probit_binned_woe_robust_scaler,0.877249,0.003758,0.692510,0.003675,0.837130,0.002506,0.563799,0.009740,0
6,logit_elastic_binned_woe,0.877158,0.003849,0.693004,0.003657,0.837771,0.002415,0.570011,0.008637,0
7,logit_binned_woe,0.877151,0.003837,0.693000,0.003681,0.837620,0.002470,0.569616,0.008344,0
8,logit_binned_woe_robust_scaler,0.877151,0.003837,0.693000,0.003681,0.837620,0.002470,0.569616,0.008344,0
9,logit_elasticnet_standard_scaler,0.867528,0.005530,0.674541,0.010970,0.828425,0.002082,0.546419,0.008390,0


In [12]:
# SLIDE_DECK_EXPORT_CLASSIFICATION
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, cross_val_predict

slide_data_dir = Path('../../slidedeck/data')
slide_data_dir.mkdir(parents=True, exist_ok=True)
output_path = slide_data_dir / 'classification_model_report.xlsx'

best_experiment = summary_df.sort_values('roc_auc_mean', ascending=False).iloc[0]['experiment']
best_pipeline = clone(experiment_grid[best_experiment])
best_pipeline.fit(X, y)

cv = skf if 'skf' in globals() else StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pred_proba = cross_val_predict(clone(experiment_grid[best_experiment]), X, y, cv=cv, method='predict_proba')[:, 1]
pred_label = (pred_proba >= 0.5).astype(int)

sample_n = min(len(X), 5000)
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X), size=sample_n, replace=False) if len(X) > sample_n else np.arange(len(X))
X_sample = X.iloc[sample_idx] if hasattr(X, 'iloc') else X[sample_idx]
y_sample = y.iloc[sample_idx] if hasattr(y, 'iloc') else y[sample_idx]
perm = permutation_importance(
    best_pipeline,
    X_sample,
    y_sample,
    n_repeats=5,
    random_state=42,
    scoring='roc_auc',
    n_jobs=1,
)
feature_names = list(X.columns) if hasattr(X, 'columns') else [f'feature_{i}' for i in range(len(perm.importances_mean))]
feature_importance_df = (
    pd.DataFrame({
        'feature': feature_names,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std,
    })
    .sort_values('importance_mean', ascending=False)
    .reset_index(drop=True)
)

prediction_export = pd.DataFrame({
    'row_id': list(X.index) if hasattr(X, 'index') else np.arange(len(pred_proba)),
    'actual_result': y.to_numpy() if hasattr(y, 'to_numpy') else np.asarray(y),
    'predicted_win_probability': pred_proba,
    'predicted_label': pred_label,
})
prediction_export['score_decile'] = pd.qcut(
    prediction_export['predicted_win_probability'].rank(method='first'),
    10,
    labels=list(range(1, 11)),
)
prediction_export['score_decile'] = prediction_export['score_decile'].astype(int)
lift_df = (
    prediction_export
    .groupby('score_decile', as_index=False)
    .agg(
        opportunities=('row_id', 'count'),
        wins=('actual_result', 'sum'),
        avg_win_probability=('predicted_win_probability', 'mean'),
        observed_win_rate=('actual_result', 'mean'),
    )
    .sort_values('score_decile')
    .reset_index(drop=True)
)
lift_df['share_of_total_wins'] = lift_df['wins'] / lift_df['wins'].sum()
lift_df['cumulative_share_of_total_wins'] = lift_df['share_of_total_wins'].cumsum()

metadata_df = pd.DataFrame([
    {
        'target': 'Opportunity Result',
        'best_experiment': best_experiment,
        'best_roc_auc_mean': summary_df.iloc[0]['roc_auc_mean'],
        'best_pr_auc_mean': summary_df.iloc[0]['pr_auc_mean'],
        'best_accuracy_mean': summary_df.iloc[0]['accuracy_mean'],
        'best_f1_mean': summary_df.iloc[0]['f1_mean'],
        'notes': 'Snapshot classification model used for prioritization use case.',
    }
])

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='cv_results', index=False)
    summary_df.to_excel(writer, sheet_name='cv_summary', index=False)
    feature_importance_df.to_excel(writer, sheet_name='feature_importance', index=False)
    prediction_export.to_excel(writer, sheet_name='oof_predictions', index=False)
    lift_df.to_excel(writer, sheet_name='prioritization_lift', index=False)
    metadata_df.to_excel(writer, sheet_name='metadata', index=False)

print('saved slide deck classification report:', output_path)
print('best experiment:', best_experiment)


/Users/goviedb/Development/mib_cars_analysis/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [00:31:12] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/Users/goviedb/Development/mib_cars_analysis/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [00:31:13] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/Users/goviedb/Development/mib_cars_analysis/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [00:31:14] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/Users/goviedb/Development/mib_cars_analysis/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [00:31:16] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/Users/goviedb/Development/mib_cars_analysis/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [00:31:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/Users/goviedb/Development/mib_cars_analysis/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [00:31:18] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


saved slide deck classification report: ../../slidedeck/data/classification_model_report.xlsx
best experiment: xgboost
